# Streamlit - Prosty Frontend w Pythonie

**Bonus:** Konsumpcja REST API bez HTML/CSS/JavaScript

---

## Spis treści

1. [Wprowadzenie](#1-wprowadzenie)
2. [Co to jest Streamlit?](#2-co-to-jest-streamlit)
3. [Jak działa Streamlit?](#3-jak-działa-streamlit)
4. [Instalacja i pierwsze uruchomienie](#4-instalacja-i-pierwsze-uruchomienie)
5. [Logi Streamlit - jak podejrzeć co się dzieje](#5-logi-streamlit-jak-podejrzeć-co-się-dzieje)
6. [Podstawy pisania aplikacji](#6-podstawy-pisania-aplikacji)
7. [Konsumpcja REST API](#7-konsumpcja-rest-api)
8. [Porównanie: HTML/JS vs Streamlit](#8-porównanie-htmljs-vs-streamlit)
9. [Best Practices](#9-best-practices)
10. [Kiedy NIE używać Streamlit](#10-kiedy-nie-używać-streamlit)
11. [Dodatkowe zasoby](#11-dodatkowe-zasoby)

---

## 1. Wprowadzenie

### Po co ten temat?

**Problem:** Stworzyłeś świetne REST API w FastAPI, ale chcesz szybko pokazać że działa.

**Tradycyjne rozwiązanie:**
- Napisz HTML (struktura)
- Napisz CSS (wygląd)
- Napisz JavaScript (logika + fetch API)
- Setup bundler (Webpack, Vite)
- Setup serwer deweloperski
- **Czas: kilka godzin** ⏰

**Rozwiązanie Streamlit:**
- Napisz jeden plik Python (~100 linii)
- Uruchom `streamlit run app.py`
- **Czas: 15 minut** ⚡

### Gdzie to użyjemy w praktyce?

- **Prototypy i MVP** - szybkie demo dla klienta
- **Internal tools** - dashboardy dla zespołu (admin panele, monitoring)
- **Data apps** - wizualizacja danych, raporty, analizy
- **ML demos** - pokazanie modeli ML bez frontendu
- **Testowanie API** - interaktywne testowanie endpointów

### Dlaczego Streamlit?

✅ **Piszesz tylko Python** - zero HTML/CSS/JavaScript  
✅ **Auto-refresh** - zmiana kodu = instant reload w przeglądarce  
✅ **Gotowe komponenty** - przyciski, inputy, tabele, wykresy  
✅ **Szybki development** - od pomysłu do działającego UI w minuty  
✅ **Idealny dla prototypów** - nie wymaga zespołu frontend dev  

❌ **NIE używaj do:**
- Aplikacji produkcyjnych dla użytkowników końcowych
- Złożonych UI (user flows, animacje, complex state)
- High-traffic apps (Streamlit nie skaluje się tak dobrze)

## 2. Co to jest Streamlit?

**Streamlit** = framework Python który **generuje HTML/CSS/JS z Twojego kodu Python** i serwuje go w przeglądarce.

### Prosty model mentalny:

```
Twój kod Python → Streamlit → HTML/CSS/JS → Przeglądarka
```

**Przykład:**
```python
st.title("Hello World")
st.button("Click me")
```

**Streamlit generuje (pod spodem):**
```html
<h1>Hello World</h1>
<button>Click me</button>
```

I serwuje to na porcie 8501!

### Analogia (dla programisty backend):

Wyobraź sobie że piszesz zwykły skrypt Python:
```python
name = input("Jak masz na imię? ")
print(f"Cześć {name}!")
```

**Streamlit robi to samo, ale w przeglądarce:**
```python
import streamlit as st

name = st.text_input("Jak masz na imię?")
st.write(f"Cześć {name}!")
```

I masz piękny UI w przeglądarce! 🎉

### Kluczowa różnica vs tradycyjny web:

| Tradycyjny web | Streamlit |
|----------------|----------|
| Piszesz HTML/CSS/JS | Piszesz Python |
| Dwa serwery (React + FastAPI) | Jeden serwer (Streamlit) |
| Ręczny state management | Auto re-run skryptu |
| fetch() / axios | requests.get() |
| DOM manipulation | st.button(), st.write() |

### Czym Streamlit **JEST**:

- ✅ Generator HTML/CSS/JS z kodu Python
- ✅ Serwer HTTP (Uvicorn na porcie 8501)
- ✅ Framework dla prototypów i MVP
- ✅ Narzędzie dla data apps i internal tools

## 3. Jak działa Streamlit?

### Architektura (uproszczona):

```
┌─────────────────────────────────────────────┐
│  Przeglądarka (http://localhost:8501)      │
│  - Wyświetla HTML/CSS/JS                   │
│  - User klika przycisk                     │
└─────────────┬───────────────────────────────┘
              │
              │ HTTP Request
              ▼
┌─────────────────────────────────────────────┐
│  Streamlit Server (localhost:8501)         │
│                                             │
│  1. Wykonuje app.py OD NOWA (cały!)        │
│  2. Jeśli w kodzie: requests.get()...      │
│     → leci do FastAPI                      │
│  3. Generuje NOWY HTML/CSS/JS               │
│  4. Wysyła do przeglądarki                 │
└─────────────┬───────────────────────────────┘
              │
              │ requests.get("http://localhost:8000/tasks")
              ▼
┌─────────────────────────────────────────────┐
│  FastAPI Server (localhost:8000)           │
│  - Zwraca JSON                              │
└─────────────────────────────────────────────┘
```

### Krok po kroku - co się dzieje:

**1. Uruchamiasz:** `streamlit run app.py`

```
Streamlit startuje serwer (Uvicorn na porcie 8501)
   ↓
Wykonuje app.py OD GÓRY DO DOŁU
   ↓
st.title() → generuje <h1>Title</h1>
st.button() → generuje <button>Click</button>
requests.get("http://localhost:8000/tasks") → leci do FastAPI
   ↓
FastAPI odpowiada: {"tasks": [...]}
   ↓
st.write() → generuje <div>Task 1</div>
   ↓
Streamlit zbiera cały HTML/CSS/JS
   ↓
Serwuje na http://localhost:8501
   ↓
Przeglądarka wyświetla stronę
```

**2. User klika przycisk:**

```
Klik w przeglądarce
   ↓
HTTP Request → Streamlit Server
   ↓
Streamlit WYKONUJE app.py OD NOWA (cały skrypt!)
   ↓
requests.get() → znowu leci do FastAPI
   ↓
FastAPI odpowiada (może nowsze dane)
   ↓
Generuje NOWY HTML/CSS/JS
   ↓
Wysyła do przeglądarki
   ↓
Przeglądarka pokazuje świeżą stronę
```

### Kluczowa zasada:

**Streamlit wykonuje CAŁY skrypt od nowa przy każdej interakcji!**

```python
import streamlit as st
import requests

# To się wykonuje ZAWSZE (przy każdym kliknięciu!)
tasks = requests.get("http://localhost:8000/v1/tasks").json()["tasks"]

# To też się wykonuje ZAWSZE
if st.button("Dodaj task"):
    # To TYLKO gdy button był kliknięty w tym re-runie
    requests.post("http://localhost:8000/v1/tasks", json={"name": "New"})
    st.rerun()  # "Hej Streamlit, uruchom skrypt od nowa!"
```

---

### Co znaczy `0.0.0.0:8501`?

Kiedy uruchamiasz Streamlit, widzisz:
```
Uvicorn server started on 0.0.0.0:8501

  Local URL:   http://localhost:8501
  Network URL: http://192.168.100.103:8501
```

**Co to oznacza?**

- **`0.0.0.0`** = "Słuchaj na WSZYSTKICH interfejsach sieciowych"
  - `127.0.0.1` (localhost) - tylko lokalnie
  - Twój IP w sieci lokalnej (np. `192.168.100.103`)
  - Dzięki temu możesz otworzyć app na telefonie w tej samej sieci WiFi!

- **`:8501`** = Port na którym Streamlit serwuje aplikację
  - Domyślnie 8501 (można zmienić: `--server.port 8502`)
  - Różny od FastAPI (8000) żeby nie kolidowały

**Dwa URL-e:**

- **Local URL (`localhost:8501`)**
  - Działa TYLKO na Twoim komputerze
  - `localhost = 127.0.0.1` (loopback)

- **Network URL (`192.168.100.103:8501`)**
  - Twój IP w sieci lokalnej
  - Inni w tej samej WiFi mogą otworzyć
  - Np. kolega z laptopa, telefon

## 4. Instalacja i pierwsze uruchomienie

### Instalacja:

```bash
pip install streamlit requests
```

- **`streamlit`** - sam framework
- **`requests`** - do konsumpcji REST API

### Pierwszy program (`hello.py`):

In [ ]:
# hello.py
import streamlit as st

st.title("Moja pierwsza aplikacja Streamlit")
st.write("Hello World! 🌍")

name = st.text_input("Jak masz na imię?")
if name:
    st.write(f"Cześć {name}! 👋")

### Uruchomienie:

```bash
streamlit run hello.py
```

**Output:**
```
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.100.103:8501
```

Przeglądarka otworzy się automatycznie! 🎉

### Co się stało?

1. **Streamlit uruchomił serwer** (Uvicorn na porcie 8501)
2. **Wykonał Twój skrypt** `hello.py` (od góry do dołu)
3. **Wygenerował HTML/CSS/JS** z `st.title()`, `st.text_input()`, itp.
4. **Zaserwował** na http://localhost:8501
5. **Otworzył przeglądarkę**
6. **Nasłuchuje zmian** - jak zapiszesz plik, UI się odświeży

---

### WAŻNE: Upewnij się że backend API działa!

**Jeśli chcesz konsumować FastAPI, musisz mieć uruchomione:**

**Terminal 1 - Backend API:**
```bash
# W katalogu app4/
fastapi dev main.py
# Backend na http://localhost:8000
```

**Terminal 2 - Streamlit Frontend:**
```bash
# W katalogu app4/
streamlit run streamlit_app.py
# Frontend na http://localhost:8501
```

**Sprawdź czy wszystko działa:**
1. Backend: http://localhost:8000 (JSON response)
2. Swagger: http://localhost:8000/docs
3. Streamlit: http://localhost:8501 (auto-open)

## 5. Logi Streamlit - jak podejrzeć co się dzieje

Domyślnie Streamlit **ukrywa logi HTTP** (access logs). Żeby zobaczyć co się dzieje:

### Sposób 1: Debug mode (najlepszy)

```bash
streamlit run streamlit_app.py --server.enableStaticServing false --logger.level debug
```

**Co zobaczysz:**
```
2026-06-19 23:46:12.734 Uvicorn server started on 0.0.0.0:8501
2026-06-19 23:46:13.123 GET /_stcore/stream HTTP/1.1 101 Switching Protocols
2026-06-19 23:46:15.456 POST /_stcore/... HTTP/1.1 200 OK
```

Każde **kliknięcie przycisku** → request do Streamlit → widzisz w logach!

### Sposób 2: Environment variable

```bash
STREAMLIT_LOG_LEVEL=debug streamlit run streamlit_app.py
```

### Sposób 3: Konfiguracja (raz na zawsze)

Stwórz plik `~/.streamlit/config.toml`:

```bash
mkdir -p ~/.streamlit
cat > ~/.streamlit/config.toml << 'EOF'
[logger]
level = "debug"

[server]
enableCORS = false
enableXsrfProtection = false
EOF
```

Potem zwykłe `streamlit run app.py` będzie pokazywało logi.

---

### Co możesz zweryfikować w logach:

**Flow requestów:**
```
1. User klika "Dodaj task" w przeglądarce
   ↓
2. Logi Streamlit: POST /_stcore/... 200 OK
   ↓
3. Logi FastAPI: POST /v1/tasks 201 Created
   ↓
4. Logi Streamlit: GET /_stcore/... 200 OK (re-render)
   ↓
5. Logi FastAPI: GET /v1/tasks 200 OK
```

**Potwierdza to model:**
- Request leci do **Streamlit**
- Streamlit wykonuje **Python kod**
- Python kod odpytuje **FastAPI** (`requests.get()`)
- FastAPI zwraca JSON
- Streamlit generuje nowy HTML
- Przeglądarka dostaje świeży HTML

## 6. Podstawy pisania aplikacji

### Podstawowe komponenty UI:

In [ ]:
import streamlit as st

# Teksty
st.title("To jest tytuł")
st.header("To jest header")
st.subheader("To jest subheader")
st.text("To jest zwykły tekst")
st.markdown("**Bold** i *italic*")

# Inputy
text = st.text_input("Wpisz tekst")
number = st.number_input("Liczba", min_value=0, max_value=100)
slider = st.slider("Slider", 0, 100, 50)
checkbox = st.checkbox("Checkbox")
radio = st.radio("Radio", ["A", "B", "C"])
select = st.selectbox("Select", ["Option 1", "Option 2"])

# Przyciski
if st.button("Kliknij"):
    st.write("Kliknięto!")

# Wyświetlanie
st.write("Cokolwiek")
st.json({"key": "value"})
st.code("print('Hello')", language="python")

# Layout
with st.container(border=True):
    st.write("Kontener")

col1, col2 = st.columns(2)
with col1:
    st.write("Kolumna 1")
with col2:
    st.write("Kolumna 2")

st.divider()

### Formularze:

In [ ]:
with st.form("my_form"):
    name = st.text_input("Imię")
    age = st.number_input("Wiek", min_value=0)
    
    submitted = st.form_submit_button("Wyślij")
    if submitted:
        st.write(f"Witaj {name}, {age} lat")

**Formularze = re-run tylko po submit (lepsza wydajność)**

### Session State:

In [ ]:
if 'counter' not in st.session_state:
    st.session_state.counter = 0

if st.button("Zwiększ"):
    st.session_state.counter += 1

st.write(f"Licznik: {st.session_state.counter}")

**Session state = utrzymywanie stanu między re-runami**

## 7. Konsumpcja REST API

### Podstawowy pattern:

In [ ]:
import streamlit as st
import requests

API_URL = "http://localhost:8000/v1"

def get_tasks():
    try:
        response = requests.get(f"{API_URL}/tasks")
        response.raise_for_status()
        return response.json()["tasks"]
    except requests.exceptions.RequestException as e:
        st.error(f"Błąd: {e}")
        return []

def create_task(name):
    try:
        response = requests.post(f"{API_URL}/tasks", json={"name": name})
        response.raise_for_status()
        return True
    except requests.exceptions.RequestException as e:
        st.error(f"Błąd: {e}")
        return False

### Kompletny przykład:

In [ ]:
import streamlit as st
import requests

API_URL = "http://localhost:8000/v1"

def get_tasks():
    return requests.get(f"{API_URL}/tasks").json()["tasks"]

def create_task(name):
    requests.post(f"{API_URL}/tasks", json={"name": name})

st.title("📋 Task Manager")

# Dodawanie
with st.form("create_form"):
    new_name = st.text_input("Nazwa")
    if st.form_submit_button("Dodaj"):
        create_task(new_name)
        st.success("Dodano!")
        st.rerun()  # Re-run → świeża lista

# Lista
st.divider()
tasks = get_tasks()
for task in tasks:
    st.write(f"**#{task['id']}** — {task['name']}")

**Flow:**
1. User klika "Dodaj"
2. `create_task()` → POST do FastAPI
3. `st.rerun()` → cały skrypt od nowa
4. `get_tasks()` → GET z FastAPI (świeże dane)
5. Przeglądarka pokazuje nowy task

## 8. Porównanie: HTML/JS vs Streamlit

### Tradycyjny sposób (HTML + JS): ~50 linii

```html
<!DOCTYPE html>
<html>
<head>
  <style>
    input { padding: 10px; }
    button { padding: 10px; background: blue; color: white; }
  </style>
</head>
<body>
  <h1>Task Manager</h1>
  <input id="taskName" placeholder="Nazwa">
  <button onclick="createTask()">Dodaj</button>
  <div id="taskList"></div>

  <script>
    async function createTask() {
      const name = document.getElementById('taskName').value;
      await fetch('http://localhost:8000/v1/tasks', {
        method: 'POST',
        body: JSON.stringify({name: name})
      });
      loadTasks();
    }

    async function loadTasks() {
      const res = await fetch('http://localhost:8000/v1/tasks');
      const data = await res.json();
      const list = document.getElementById('taskList');
      list.innerHTML = '';
      data.tasks.forEach(task => {
        const div = document.createElement('div');
        div.textContent = `#${task.id} — ${task.name}`;
        list.appendChild(div);
      });
    }
    loadTasks();
  </script>
</body>
</html>
```

**3 języki (HTML, CSS, JS) • DOM manipulation • fetch API • async/await**

---

### Streamlit: ~15 linii

```python
import streamlit as st
import requests

API = "http://localhost:8000/v1"

st.title("Task Manager")

with st.form("add"):
    name = st.text_input("Nazwa")
    if st.form_submit_button("Dodaj"):
        requests.post(f"{API}/tasks", json={"name": name})
        st.rerun()

tasks = requests.get(f"{API}/tasks").json()["tasks"]
for task in tasks:
    st.write(f"#{task['id']} — {task['name']}")
```

**1 język (Python) • Zero DOM • requests.get() • prosty flow**

---

### Porównanie:

| Aspekt | HTML/CSS/JS | Streamlit |
|--------|-------------|----------|
| Linie kodu | ~50 | ~15 |
| Języki | 3 | 1 |
| Czas | Godziny | Minuty |
| Learning curve | Wysoka | Niska |
| Produkcja | ✅ | ❌ |
| Prototypy | ❌ | ✅ |

## 9. Best Practices

### ✅ Rób tak:

**1. Funkcje dla API:**
```python
def get_tasks():
    return requests.get(f"{API}/tasks").json()["tasks"]
```

**2. Obsługa błędów:**
```python
try:
    tasks = get_tasks()
except:
    st.error("Błąd API")
    tasks = []
```

**3. Cache dla ciężkich operacji:**
```python
@st.cache_data(ttl=60)
def load_data():
    return requests.get(f"{API}/data").json()
```

**4. st.rerun() po modyfikacji:**
```python
if st.button("Dodaj"):
    create_task(name)
    st.rerun()  # Świeża lista
```

### ❌ Nie rób tak:

**1. Requesty bez cache:**
```python
# ❌ Każdy re-run = nowy request
tasks = requests.get(f"{API}/tasks").json()
```

**2. Globalne zmienne dla stanu:**
```python
# ❌ Będzie resetowane przy re-run
counter = 0

# ✅ Session state
if 'counter' not in st.session_state:
    st.session_state.counter = 0
```

## 10. Kiedy NIE używać Streamlit

### ❌ NIE używaj do:

1. **Aplikacji produkcyjnych** dla użytkowników końcowych
   - Nie skaluje się przy dużym ruchu
   - Każdy user = osobna sesja Python

2. **Złożonych UI** z routingiem
   - Brak zaawansowanego routingu (jak React Router)
   - Brak nested routes, transitions

3. **Real-time apps** (chat, live notifications)
   - Lepiej Socket.io + React

4. **Aplikacji offline**
   - Wymaga connection do serwera

### ✅ Używaj do:

1. **Prototypów i MVP**
2. **Internal tools** (admin, monitoring)
3. **Data apps** (dashboardy, raporty)
4. **Testowania API**
5. **Edukacji** (jak to!)

## 11. Dodatkowe zasoby

### Dokumentacja:
- [Streamlit Docs](https://docs.streamlit.io/)
- [API Reference](https://docs.streamlit.io/library/api-reference)
- [Gallery](https://streamlit.io/gallery)

### Deployment:
- [Streamlit Cloud](https://streamlit.io/cloud) - darmowy hosting
- [Docker](https://docs.streamlit.io/knowledge-base/tutorials/deploy/docker)

### Community:
- [Forum](https://discuss.streamlit.io/)
- [GitHub](https://github.com/streamlit/streamlit)

---

## Podsumowanie

**Streamlit:**
- 🐍 Generuje HTML/CSS/JS z kodu Python
- 🚀 Serwer HTTP (Uvicorn:8501)
- 🔄 Re-run całego skryptu przy interakcji
- ⚡ Świetny do prototypów, MVP, internal tools
- ❌ Nie dla produkcji użytkowników końcowych

**Flow:**
```
Python → Streamlit → HTML/CSS/JS → Przeglądarka
   ↓                                    ↓
requests.get() → FastAPI ← HTTP Request
```

**Logi:**
```bash
streamlit run app.py --server.enableStaticServing false --logger.level debug
```

**Następny krok:** Sprawdź `streamlit_app.py` w `app4/` - gotowa aplikacja! 🎉